<a href="https://colab.research.google.com/github/RevanthSettipalli/Hydrology-Groundwater-Quality-Prediction/blob/main/Ground_Water.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -----------------------------
# INSTALL LIBRARIES
# -----------------------------

!pip install -q folium branca plotly pytorch-tabnet torch torchvision scikit-learn

# -----------------------------
# IMPORT LIBRARIES
# -----------------------------
from IPython.display import HTML
from google.colab import files
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import folium
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.ensemble import GradientBoostingClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

# =============================================================
# LOAD DATASET
# =============================================================
uploaded = files.upload()
filename = list(uploaded.keys())[0]
data = pd.read_csv(io.BytesIO(uploaded[filename]), encoding='latin1')

print("✅ Dataset Loaded")

# =============================================================
# DATA CLEANING
# =============================================================
for col in ['pH','TH','Ca']:
    data[col] = pd.to_numeric(data[col], errors='coerce')

data.fillna(data.mean(numeric_only=True), inplace=True)

# =============================================================
# LABEL CREATION
# =============================================================
def label_water(ph, hardness, chemical):
    if 6.5 <= ph <= 8.5 and hardness < 200 and chemical < 75:
        return 0
    elif hardness < 400 and chemical < 100:
        return 1
    else:
        return 2

data['Label'] = data.apply(lambda row: label_water(row['pH'], row['TH'], row['Ca']), axis=1)

# =============================================================
# TRAIN TEST SPLIT
# =============================================================
X = data[['pH','TH','Ca']]
y = data['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Torch tensors
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)

# =============================================================
# PROFESSIONAL MODEL TRAINING REPORT
# =============================================================

print("\n" + "="*70)
print("GROUNDWATER QUALITY MODEL TRAINING REPORT")
print("="*70)

print("\nDataset Summary")
print("Total Samples:", len(data))
print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nFeatures Used")
print("• pH")
print("• TH")
print("• Ca")

print("\nTraining Method")
print("1. Data Cleaning")
print("2. Feature Scaling")
print("3. Train/Test Split")
print("4. Independent Training")
print("5. Accuracy Evaluation")

print("\nTraining Started")
print("="*70)


# ----------------------
# TabNet
# ----------------------

print("\nTraining Model → TabNet")

tabnet = TabNetClassifier(
    verbose=0
)

tabnet.fit(
    X_train_s,
    y_train,
    max_epochs=20
)

print("Status → Completed")


# ----------------------
# DeepGBM
# ----------------------

print("\nTraining Model → DeepGBM")

gbm = GradientBoostingClassifier()

gbm.fit(
    X_train_s,
    y_train
)

print("Status → Completed")


# ----------------------
# Neural Model
# ----------------------

class NN(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(3,64),
            nn.ReLU(),

            nn.Linear(64,32),
            nn.ReLU(),

            nn.Linear(32,3)

        )

    def forward(self,x):

        return self.net(x)


def train_nn(model):

    opt = torch.optim.Adam(
        model.parameters(),
        lr=0.01
    )

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(20):

        opt.zero_grad()

        loss = loss_fn(
            model(
                X_train_t
            ),
            y_train_t
        )

        loss.backward()

        opt.step()

    return model


print("\nTraining Model → TabTransformer")

tt_model = train_nn(
    NN()
)

print("Status → Completed")


print("\nTraining Model → FT-Transformer")

ft_model = train_nn(
    NN()
)

print("Status → Completed")


# =============================================================
# PERFORMANCE REPORT
# =============================================================

results = {

"TabNet":

accuracy_score(
y_test,
tabnet.predict(
X_test_s
)
),

"DeepGBM":

accuracy_score(
y_test,
gbm.predict(
X_test_s
)
),

"TabTransformer":

accuracy_score(
y_test,
torch.argmax(
tt_model(
X_test_t
),
1
).numpy()
),

"FT-Transformer":

accuracy_score(
y_test,
torch.argmax(
ft_model(
X_test_t
),
1
).numpy()
)

}

print("\n")
print("="*70)
print("MODEL PERFORMANCE")
print("="*70)

for model, score in results.items():

    print(
        model,
        "→",
        round(
            score*100,
            2
        ),
        "%"
    )


best_model = max(
results,
key=results.get
)

print("\nSelected Model")

print(
best_model
)

print(
"Final Accuracy:",
round(
results[
best_model
]*100,
2
),
"%"
)

print("="*70)

# =============================================================
# EVALUATION (CONFUSION + F1 + ROC)
# =============================================================
if best_model == "TabNet":
    y_pred = tabnet.predict(X_test_s)
    y_score = tabnet.predict_proba(X_test_s)
elif best_model == "DeepGBM":
    y_pred = gbm.predict(X_test_s)
    y_score = gbm.predict_proba(X_test_s)
else:
    probs = torch.softmax(tt_model(X_test_t), dim=1).detach().numpy()
    y_pred = np.argmax(probs, axis=1)
    y_score = probs

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i][j], ha='center', va='center')
plt.show()

# Classification Report
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# ROC Curve
y_test_bin = label_binarize(y_test, classes=[0,1,2])

plt.figure()
for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    plt.plot(fpr, tpr, label=f"Class {i}")

plt.plot([0,1],[0,1],'k--')
plt.title("ROC Curve")
plt.legend()
plt.show()

# =============================================================
# FUNCTIONS
# =============================================================
label_map = {0:"SAFE",1:"MODERATE",2:"HIGHLY CONTAMINATED"}

def predict_best(s_scaled, s_t):
    if best_model == "TabNet":
        return tabnet.predict(s_scaled)[0]
    elif best_model == "DeepGBM":
        return gbm.predict(s_scaled)[0]
    else:
        return torch.argmax(tt_model(s_t),1).item()

def calculate_wqi(ph, th, ca):
    score = (abs(7-ph)*20) + (th/10) + ca
    if score < 50: return score,"Excellent"
    elif score <100: return score,"Good"
    elif score <200: return score,"Poor"
    else: return score,"Unsafe"


# =============================================================
# LOCATION SYSTEM
# =============================================================

while True:

    print("\n" + "="*60)
    print("GROUNDWATER LOCATION PREDICTION")
    print("="*60)

    loc = input(
        "\nEnter LOCATION "
        "(or type EXIT): "
    )

    if loc.lower() == "exit":

        print("\nSession Closed")
        break


    area = data[
        data[
            'LOCATION'
        ].str.lower()
        ==
        loc.lower()
    ]


    if area.empty:

        print(
            "\n❌ Location not found"
        )

        continue


    row = area.iloc[0]

    ph = row['pH']
    th = row['TH']
    ca = row['Ca']

    lat = row['LATITUDE']
    lon = row['LONGITUDE']


    sample = [[ph, th, ca]]

    s_scaled = scaler.transform(
        sample
    )

    s_t = torch.tensor(
        s_scaled,
        dtype=torch.float32
    )


    print("\n📊 PARAMETERS")

    print(
        "pH:",
        ph
    )

    print(
        "Hardness:",
        th
    )

    print(
        "Chemical:",
        ca
    )


    best_pred = predict_best(
        s_scaled,
        s_t
    )

    print(
        "\n🏆 Prediction:",
        label_map[
            best_pred
        ]
    )


    wqi, status = calculate_wqi(
        ph,
        th,
        ca
    )

    print(
        "\n💧 WQI:",
        round(
            wqi,
            2
        ),
        "-",
        status
    )


    risk = {

        "SAFE":20,

        "MODERATE":50,

        "HIGHLY CONTAMINATED":90

    }


    fig = go.Figure(

        go.Indicator(

            mode="gauge+number",

            value=
            risk[
                label_map[
                    best_pred
                ]
            ],

            title={
                'text':
                "Water Risk Meter"
            }

        )

    )

    fig.show()


    plt.figure()

    plt.bar(

        [
            'pH',
            'Hardness',
            'Chemical'
        ],

        [
            ph,
            th,
            ca
        ]

    )

    plt.title(
        "Water Parameters - " +
        loc
    )

    plt.show()


    # MAP
    import plotly.express as px

    map_df = pd.DataFrame({

        "Location":[loc],

        "Latitude":[
            float(lat)
        ],

        "Longitude":[
            float(lon)
        ],

        "Status":[
            label_map[
                best_pred
            ]
        ]

    })


    fig = px.scatter_geo(

        map_df,

        lat="Latitude",

        lon="Longitude",

        hover_name="Location",

        color="Status",

        projection="natural earth",

        title=
        "Water Quality Location"

    )

    fig.show()


    print("\n" + "="*60)

    choice = input(
        "\nSearch another location? "
        "(Y/N): "
    )


    if choice.lower() != "y":

        print(
            "\nProgram Ended"
        )

        break
    # =============================================================
    # MAP (GOOGLE COLAB FINAL)
    # =============================================================

    import plotly.express as px
    import pandas as pd

    map_df = pd.DataFrame({
        "Location":[loc],
        "Latitude":[float(lat)],
        "Longitude":[float(lon)],
        "Status":[label_map[best_pred]]
    })

    fig = px.scatter_geo(
        map_df,
        lat="Latitude",
        lon="Longitude",
        hover_name="Location",
        color="Status",
        projection="natural earth",
        title=f"Water Quality Location → {loc}"
    )

    fig.update_geos(
        showcountries=True,
        countrycolor="black",
        showcoastlines=True,
        coastlinecolor="black",
        showland=True
    )

    fig.update_layout(
        height=700
    )

    fig.show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.3 MB/s eta 0:00:00
